# 🎯 Portfolio Optimization & Efficient Frontier
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Realosunboy6/free-portfolio-visualizer/blob/main/notebooks/02_optimization.ipynb)

Every optimizer Portfolio Visualizer offers — and the paid-only ones — free: GMV, Max Sharpe, target return, efficient frontier, risk parity, min CVaR, max Sortino, Kelly, Omega, min max-drawdown, tracking error, Black-Litterman, Michaud resampling. Ledoit-Wolf shrinkage by default (PV doesn't do this).

In [ ]:
#@title Setup — run this first {display-mode: "form"}
try:
    import portlab
except ImportError:
    %pip install -q "portlab @ git+https://github.com/Realosunboy6/free-portfolio-visualizer.git"
    import portlab
print("portlab", portlab.__version__, "ready")

In [ ]:
#@title Settings {display-mode: "form"}
tickers = "VTI, VEA, VWO, TLT, IEF, GLD, VNQ"  #@param {type:"string"}
start_date = "2012-01-01"   #@param {type:"date"}
end_date = ""               #@param {type:"string"}
risk_free_rate = 0.03       #@param {type:"number"}
max_weight_per_asset = 1.0  #@param {type:"number"}
covariance_estimator = "ledoit_wolf"  #@param ["ledoit_wolf", "sample", "ewma"]
SMOKE = False

In [ ]:
import numpy as np, pandas as pd
from portlab import optimize as opt, plots
from portlab.covariance import corr_from_cov, get_cov
from portlab.data import get_returns

tick_list = [t.strip().upper() for t in tickers.split(",") if t.strip()]
rets = get_returns(tick_list, start_date, end_date or None)
mu = rets.mean() * 252
cov = get_cov(rets, method=covariance_estimator)
bounds = (0.0, max_weight_per_asset)

portfolios = {
    "Equal Weight": pd.Series(1/len(mu), index=mu.index),
    "GMV": opt.gmv(mu, cov, bounds=bounds),
    "Max Sharpe": opt.max_sharpe(mu, cov, rf=risk_free_rate, bounds=bounds),
    "Risk Parity": opt.equal_risk_contribution(cov, bounds=bounds),
    "Min CVaR 95%": opt.min_cvar(rets, bounds=bounds),
    "Max Sortino": opt.max_sortino(rets, rf=risk_free_rate, bounds=bounds),
    "Kelly": opt.kelly(rets, bounds=bounds),
    "Max Omega": opt.max_omega(rets, bounds=bounds),
    "Min MaxDrawdown": opt.min_max_drawdown(rets, bounds=bounds),
}
weights_table = pd.DataFrame(portfolios)
weights_table.style.format("{:.1%}")

In [ ]:
stats = pd.DataFrame({n: opt.portfolio_stats(w.values, mu, cov, risk_free_rate)
                      for n, w in portfolios.items()}).T
fr = opt.frontier(mu, cov, n_points=40, rf=risk_free_rate, bounds=bounds)
plots.frontier_chart(fr, mu, cov,
    highlight={n: (s["volatility"], s["return"]) for n, s in stats.iterrows()
               if n in ("GMV", "Max Sharpe", "Risk Parity")}).show()
stats.style.format("{:.3f}")

In [ ]:
plots.corr_heatmap(corr_from_cov(cov)).show()
plots.weights_chart(portfolios["Max Sharpe"], "Max Sharpe Allocation").show()

In [ ]:
#@title Black-Litterman: blend market equilibrium with YOUR views {display-mode: "form"}
#@markdown Example view — first asset outperforms its equilibrium return:
view_asset = "VTI"          #@param {type:"string"}
view_annual_return = 0.09   #@param {type:"number"}
view_confidence = 0.5       #@param {type:"number"}
mkt = pd.Series(1.0, index=mu.index)  # equal caps as neutral prior; replace with market caps
P = pd.DataFrame(np.zeros((1, len(mu))), columns=mu.index)
P.loc[0, view_asset.strip().upper()] = 1.0
bl_mu = opt.black_litterman(cov, mkt, P, pd.Series([view_annual_return]),
                            view_confidence=pd.Series([view_confidence]), rf=risk_free_rate)
w_bl = opt.max_sharpe(bl_mu, cov, rf=risk_free_rate, bounds=bounds)
pd.DataFrame({"Prior mu": opt.implied_returns(cov, mkt, rf=risk_free_rate),
              "BL mu": bl_mu, "BL Max Sharpe": w_bl}).style.format("{:.2%}")

In [ ]:
# Michaud resampled frontier — robust to estimation error (PV charges for robustness)
w_rs = opt.resampled_weights(rets, n_samples=60 if not SMOKE else 8)
pd.DataFrame({"Max Sharpe (point estimate)": portfolios["Max Sharpe"],
              "Resampled Max Sharpe": w_rs}).style.format("{:.1%}")

In [ ]:
# Geometric mean frontier (Bernstein & Wilkinson) — maximizes compound growth
gf = opt.geometric_frontier(rets, n_points=10 if not SMOKE else 4, bounds=bounds)
plots.rolling_chart(gf.set_index("realized_vol")["geometric_mean"],
                    "Geometric Mean Frontier", yformat=".1%").show()
gf[["vol_cap", "realized_vol", "geometric_mean"]].style.format("{:.2%}")